# How to use vLLM as a LLM service

## Step 1: Launch vLLM Server
With the container ready, the next step is launching the vLLM server. Please **open a new terminal** to run following script:

```bash
export TP=1
export MODEL_ID="openai/gpt-oss-120b"
export VLLM_ROCM_USE_AITER=1
export VLLM_USE_AITER_UNIFIED_ATTENTION=1
export VLLM_ROCM_USE_AITER_MHA=0

vllm serve \
    $MODEL_ID \
    --port 8000 \
    --tensor-parallel $TP \
    --no-enable-prefix-caching \
    --disable-log-requests \
    --compilation-config '{"full_cuda_graph": true}' 
```

On MI300 GPUs, we enable AITER kernels and unified attention. We then start the GPT-OSS 120B model using the vllm serve command. Key options include `--tensor-parallel` to define GPU count, and `--compilation-config` to manage graph compilation and sequence length. Prefix caching and logging can be toggled to cut overhead. Once launched, your GPU runs a production-ready inference server, essentially turning it into a private mini-ChatGPT.


After a few minutes, if you have logs as below, then congrats, vLLM server is successfully on now!
```text
INFO:     Started server process [1022120]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
```

## Step 2: Send a request to the launched server
To test the vLLM server, send an HTTP request using curl. Specify the model, a prompt like `The future of AI is` and parameters such as `max_tokens` and `temperature`. The server processes the request on the GPU and returns generated text. This confirms the server is operational and optimized, providing a functional inference API for integration into applications.

In [1]:
!curl http://localhost:8000/v1/completions \
    -H "Content-Type: application/json" \
    -d '{  \
        "model": "openai/gpt-oss-120b",  \
        "prompt": "The future of AI is",  \
        "max_tokens": 100,  \
        "temperature": 0  \
    }'  

{"id":"cmpl-c39864d7b86943e7b68cef35b605d92e","object":"text_completion","created":1775792456,"model":"openai/gpt-oss-120b","choices":[{"index":0,"text":" bright, but it also presents challenges. As we continue to develop and integrate AI into our lives, we must be mindful of the potential risks and work towards creating a future where AI benefits everyone.\n\nHere are some suggestions for improving the article:\n\n1. Add a hook: The article could benefit from a hook that captures the reader's attention and draws them in. This could be a surprising statistic or a compelling anecdote that illustrates the impact of AI on society.\n2. Provide more specific examples:","logprobs":null,"finish_reason":"length","stop_reason":null,"prompt_logprobs":null}],"service_tier":null,"system_fingerprint":null,"usage":{"prompt_tokens":5,"total_tokens":105,"completion_tokens":100,"prompt_tokens_details":null},"kv_transfer_params":null}

# How to Benchmark vLLM Performance



## vllm bench serve 
This command tests online serving performance under concurrent loads (e.g., 32 users) using synthetic random data. It measures real-world metrics like Time to First Token (TTFT), Time Per Output Token (TPOT), Inter-Token Latency (ITL), and End-to-End Latency (E2EL) to evaluate responsiveness and stability in API-like environments. Before running command below, you should firstly launch vLLM server following [Launch vLLM Server](#step-1-launch-vllm-server)

In [2]:
MODEL_ID = "openai/gpt-oss-120b"
CONTAINER_NAME="vllm-demo"

!vllm bench serve \
    --model {MODEL_ID} \
    --dataset-name random \
    --dataset-name random \
    --random-input-len 4096 \
    --random-output-len 1024 \
    --max-concurrency 8 \
    --num-prompts 80 \
    --ignore-eos \
    --percentile_metrics ttft,tpot,itl,e2el

INFO 04-10 03:41:12 [__init__.py:235] Automatically detected platform rocm.
Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7f2e5e09b560>, seed=0, num_prompts=80, dataset_name='random', no_stream=False, dataset_path=None, custom_output_len=256, custom_skip_chat_template=False, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, random_input_len=4096, random_output_len=1024, random_range_ratio=0.0, random_prefix_len=0, hf_subset=None, hf_split=None, hf_output_len=None, endpoint_type='openai', label=None, backend='vllm', base_url=None, host='127.0.0.1', port=8000, endpoint='/v1/completions', max_concurrency=8, model='openai/gpt-oss-120b', tokenizer=None, use_beam_search=False, logprobs=None, request_rate=inf, burstiness=1.0, trust_remote_code=False, disable_tqdm=False, profile=False, save_result=False, save_detailed=False, append_result=False, metadata=None, result_dir=None, result_

`--max-concurrency`: This defines the maximum number of requests that can be actively processed by the server at the same time. It acts like a gatekeeper, controlling the level of parallel load on the system. 

`--num-prompts`: This is the total number of requests (prompts) that the benchmark will send. 

The recommendation to set num-prompts to 10 times the max-concurrency value is a best practice for achieving statistically reliable and stable performance metrics during a benchmark test. It’s better to set num-prompts as 10 times of max-concurrency value. The general setting of `--max-concurrency` for MI300X is [128 64 32 16 8]. 

## vllm bench latency

Since `vllm bench latency` and `vllm bench throughput` will launch a new vllm service for benchmark, before we advance, please run following command to kill running vLLM service.

In [3]:
!pkill -9 -f VLLM

 
Evaluates single-request performance without parallelism (`tensor-parallel=1`). It measures prompt processing and token generation latency for individual requests, highlighting baseline performance without concurrency or batching effects, ideal for low-load scenarios.

In [4]:
MODEL_ID = "openai/gpt-oss-120b"
CONTAINER_NAME="vllm-demo"

!vllm bench latency \
    --model {MODEL_ID} \
    --input-len 4096 \
    --output-len 1024 \
    --tensor-parallel 1 

INFO 04-10 03:45:22 [__init__.py:235] Automatically detected platform rocm.
Parse safetensors files: 100%|█████████████████| 15/15 [00:00<00:00, 181.86it/s]
INFO 04-10 03:46:03 [config.py:1605] Using max model len 131072
WARNING 04-10 03:46:03 [config.py:1085] mxfp4 quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 04-10 03:46:03 [config.py:2416] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-10 03:46:07 [__init__.py:235] Automatically detected platform rocm.
INFO 04-10 03:46:22 [core.py:581] Waiting for init message from front-end.
INFO 04-10 03:46:22 [core.py:73] Initializing a V1 LLM engine (v0.1.dev8023+g894bed8) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load

## vllm bench throughput

Since `vllm bench latency` and `vllm bench throughput` will launch a new vllm service for benchmark, before we advance, please run following command to kill running vLLM service.

In [5]:
!pkill -9 -f VLLM

Focuses on maximizing token generation speed under hardware optimization (e.g., 4-way tensor parallelism). It stresses batch processing efficiency by generating fixed-length outputs, measuring raw tokens/second, and ignoring concurrency dynamics to benchmark peak computational throughput.

Batch processing: Batch processing groups multiple requests together to be processed simultaneously by the AI. This maximizes hardware efficiency and increases overall throughput, but it can increase the latency for any individual request. It is ideal for non-interactive, high-volume tasks. 

`--tensor-parallel n` which `n` value set the card number we adopt to serve the model.

In [6]:
MODEL_ID = "openai/gpt-oss-120b"
CONTAINER_NAME="vllm-demo"

!vllm bench throughput \
    --model {MODEL_ID} \
    --dataset-name random \
    --input-len 4096 \
    --output-len 1024 \
    --num-prompts 4 \
    --tensor-parallel 1 

INFO 04-10 04:01:51 [__init__.py:235] Automatically detected platform rocm.
When dataset path is not set, it will default to random dataset
INFO 04-10 04:02:05 [datasets.py:355] Sampling input_len from [4096, 4096] and output_len from [1024, 1024]
Parse safetensors files: 100%|█████████████████| 15/15 [00:00<00:00, 194.16it/s]
INFO 04-10 04:02:25 [config.py:1605] Using max model len 131072
WARNING 04-10 04:02:25 [config.py:1085] mxfp4 quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 04-10 04:02:26 [config.py:2416] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-10 04:02:30 [__init__.py:235] Automatically detected platform rocm.
INFO 04-10 04:02:42 [core.py:581] Waiting for init message from front-end.
INFO 04-10 04:02:42 [core.py:73] Initializing a V1 LLM engine (v0.1.dev8023+g894bed8) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokeni